## 🧠🔍 Semantic Search On Books Dataset

### Using FAISS (Facebook AI Similarity Search)

In [11]:
!pip install faiss-cpu

In [12]:
!pip install datasets

#### Loading, filtering and slicing dataset

In [13]:
from datasets import load_dataset

# loading goodreads book dataset having 1M+ rows
books_dataset = load_dataset('booksouls/goodreads-book-descriptions', split='train')

books_dataset

Dataset({
    features: ['title', 'description'],
    num_rows: 1021106
})

In [14]:
# Analysing size of decription of books

books_dataset.set_format('pandas')
df = books_dataset[:]

df['length'] = df['description'].str.len()

print(df['length'].describe())

count    1.021106e+06
mean     8.557749e+02
std      5.863045e+02
min      1.000000e+00
25%      4.660000e+02
50%      7.690000e+02
75%      1.106000e+03
max      6.489700e+04
Name: length, dtype: float64


In [17]:
from datasets import Dataset

books_dataset = Dataset.from_pandas(df)

# filtering books with 300-500 sized description
sliced_dataset = books_dataset.filter(lambda x: x['length']>300 and x['length']<500)

sliced_dataset

Filter:   0%|          | 0/1021106 [00:00<?, ? examples/s]

Dataset({
    features: ['title', 'description', 'length'],
    num_rows: 141806
})

**Using 2000 books to start with as a PoW**

In [19]:
sliced_dataset = sliced_dataset.shuffle().select(range(2000))

sliced_dataset

Dataset({
    features: ['title', 'description', 'length'],
    num_rows: 2000
})

In [20]:
samples = sliced_dataset.shuffle().select(range(3))

for sample in samples:
  print("Title: ", sample['title'])
  print("Description: ", sample['description'])

Title:  Dragons and Dragon Lore
Description:  Fascinating book teems with information about powerful serpents of the deep and land-roving, fire-breathing monsters that first appeared in the creation myths of the ancient Far East. Dragons in China, Korea, and Japan are covered, as are those in Babylonian and Egyptian legends, and in English, Irish, and French tales.
Title:  In at the Deep End: Cooking Fish Venice to Tokyo
Description:  Travelling from Venice to Tokyo, New York to Sweden and Aberdeen to Sydney, Jake chronicles his journey. Whether cooking and eating Venetian bigoli with clams or New York crab cakes, Swedish soused herrings or Japanese sushi and sashimi, he effortlessly conjures up the worlds in which these dishes originated.
Title:  The Ice's Edge: The Story of a Harp Seal Pup
Description:  On the pack ice off the coast of Labrador, Little Harp Seal is born under a cloudy March sky. His home os fraught with danger. Little Harp Seal's fluffy fur hides him from prowling po

In [21]:
# Let us save the csv

sliced_dataset.to_csv('books.csv')

Creating CSV from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

885384

#### Pre-processing before generating embedding

In [22]:
# Searching based on title as well as description
def concatenate_text(examples):
    """
    Concatenate title and description.
    """

    return {
        "text": examples["title"]
        + " \n "
        + examples["description"]
    }


mod_books_dataset = sliced_dataset.map(concatenate_text)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [23]:
from transformers import AutoTokenizer, AutoModel

# we will use mpnet for embedding generation
checkpoint = "sentence-transformers/multi-qa-mpnet-base-dot-v1"       # produces 768 dimensional embedding

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [24]:
# Using CLS pooling for embedding generation

def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]    # embedding of [CLS] token

In [25]:
def get_embeddings(text_list):

    encoded_input = tokenizer(
        text_list,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    model_output = model(**encoded_input)

    return cls_pooling(model_output)

In [26]:
# generating embedding for a example

emb = get_embeddings(mod_books_dataset[0]['text'])

In [27]:
emb.shape

torch.Size([1, 768])

#### Generating mpnet embedding of entire dataset

In [28]:
# generating the dataset with embedding
# FAISS excepts embedding in numpy arrays

embedding_dataset = mod_books_dataset.map(
    lambda x: {"embeddings": get_embeddings(x["text"]).detach().cpu().numpy()[0]}
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [29]:
embedding_dataset

Dataset({
    features: ['title', 'description', 'length', 'text', 'embeddings'],
    num_rows: 2000
})

In [31]:
# FAISS indexing to embeddings
embedding_dataset.add_faiss_index(column="embeddings")

  0%|          | 0/2 [00:00<?, ?it/s]

Dataset({
    features: ['title', 'description', 'length', 'text', 'embeddings'],
    num_rows: 2000
})

#### Querying on the dataset

In [32]:
# generating Question embeddings for querying
q = "Find the book with Xavier Delacroix and the fall of the Great Vampire"
question_embedding = get_embeddings([q]).cpu().detach().numpy()
question_embedding.shape

(1, 768)

In [33]:
scores, samples = embedding_dataset.get_nearest_examples(
    "embeddings", question_embedding, k=3
)

In [39]:
samples['title']

["The Immortal's Guide",
 'Of Masques and Martyrs (Shadow Saga #3)',
 'Saragossa - The Vampire Legacy']

In [40]:
scores

array([26.32698 , 36.931343, 40.694393], dtype=float32)

In [42]:
# let's try another example

sample = sliced_dataset.shuffle().select(range(1))

print(sample['title'])
print(sample['description'])

['Around the World: A Follow-the-Trail Book']
['In this interactive novelty board book, little ones trace a die-cut trail to explore the world.\nUse your finger to help different creatures find their special trail around the world! This interactive board book lets little ones explore world by tracing a tactile pathway. Each spread will feature a different animal looking for its way through a different landscape, from a thick, green forest to a dry, orange desert.']


In [43]:
q = "Interactive board book for little ones to explore the world"
question_embedding = get_embeddings([q]).cpu().detach().numpy()

scores, samples = embedding_dataset.get_nearest_examples(
    "embeddings", question_embedding, k=5
)

print(samples['title'])
print(scores)

['Around the World: A Follow-the-Trail Book', 'My Learning Library Box Set', 'Bedtime Stories: 12 Board Book Block (Disney: Book Block)', 'The Book With a Hole', 'One Little Bird and Her Friends: A counting board book']
[26.029972 30.840563 33.39431  34.498825 34.78506 ]


In [48]:
sample = sliced_dataset.shuffle().select(range(1))

print(sample['title'])
print(sample['description'])

['The Essential Dracula']
["Here is the complete original text of Bram Stoker's classic 1897 novel, fully annotated with thousands of fascinating facts. Includes: background on Stoker's classic, and the literary history of the vampire novel; commentary by leading contemporary writers; a selected filmography of major vampire films; and dozens of illustrations."]


In [50]:
q = "History of Vampire related books"              # general question, might fetch different books
question_embedding = get_embeddings([q]).cpu().detach().numpy()

scores, samples = embedding_dataset.get_nearest_examples(
    "embeddings", question_embedding, k=5
)

print(samples['title'])
print(scores)

['Saragossa - The Vampire Legacy', 'Of Masques and Martyrs (Shadow Saga #3)', 'The Essential Dracula', 'Vampyre Sanguinomicon: The Lexicon of the Living Vampire', 'The Encyclopedia of Demons and Demonology']
[34.60187  34.776382 38.920944 40.317772 42.075706]


#### Saving FAISS indices

In [47]:
mod_books_dataset.save_to_disk("books_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/2000 [00:00<?, ? examples/s]

In [46]:
# saving faiss indexes separately
embedding_dataset.get_index("embeddings").save("books_dataset_faiss.index")